In [17]:
import cv2
import numpy as np
import os

In [18]:
def create_sketch_mask(img_gray, k_median_size, k_laplace_size, threshold_value):
    """
    Creates a black-and-white sketch mask based on your provided code.
    Steps:
    1. Grayscale image is provided as input.
    2. Apply median filter for noise reduction.
    3. Apply Laplacian filter for edge detection.
    4. Normalize the Laplacian output to 0-255 range.
    5. Apply binary threshold.
    """
    
    print("Applying Median Filter...")
    img_median_blur = cv2.medianBlur(img_gray, k_median_size)

    
    print("Applying Laplacian Filter...")
    img_edge = cv2.Laplacian(img_median_blur, cv2.CV_16S, ksize=k_laplace_size)

    
    img_edge_scaled = cv2.normalize(img_edge, None, alpha=0, beta=255, norm_type=cv2.NORM_MINMAX, dtype=cv2.CV_8U)

   
    print("Applying Threshold...")
    _ , img_thresh = cv2.threshold(img_edge_scaled, threshold_value, 255, cv2.THRESH_BINARY_INV)
    
    return img_thresh

In [ ]:
def create_color_painting(image, d_size, sigma_color, sigma_space, repetition_count, downsample_factor=0.5):
    """
    Creates the 'color painting' effect using an optimized bilateral filter.
    """
    print("Creating Color Painting...")
    original_height, original_width = image.shape[:2]
    
    small_img = cv2.resize(image, (0, 0), fx=downsample_factor, fy=downsample_factor, interpolation=cv2.INTER_LINEAR)
    
    smoothed_img = small_img
    for _ in range(repetition_count):
        smoothed_img = cv2.bilateralFilter(smoothed_img, d_size, sigma_color, sigma_space)
    
    color_painting = cv2.resize(smoothed_img, (original_width, original_height), interpolation=cv2.INTER_LINEAR)
    
    return color_painting

In [ ]:
def combine_with_sketch(color_painting, sketch_mask):
    """
    Overlays the black sketch lines onto the color painting.
    """
    print("Combining painting and sketch...")
    
    target_height, target_width = color_painting.shape[:2]
    
    
    resized_mask = cv2.resize(sketch_mask, (target_width, target_height))

    
    final_output = cv2.bitwise_and(color_painting, color_painting, mask=resized_mask)
    
    return final_output

In [ ]:
def main():
    
    INPUT_IMAGE_PATH = "orgImg.png" 
    
    
    # For the Sketch Mask 
    MEDIAN_KERNEL_SIZE = 5
    LAPLACIAN_KERNEL_SIZE = 7
    THRESHOLD_VALUE = 140
    
    # For the Color Painting
    DOWNSAMPLE_FACTOR = 0.5
    REPETITION_COUNT = 5    
    D_SIZE = 9            
    SIGMA_COLOR = 15      
    SIGMA_SPACE = 15      

    
    
    print("Loading images...")
    original_image = cv2.imread(INPUT_IMAGE_PATH)
    if original_image is None:
        print(f"Error: Could not load image from '{INPUT_IMAGE_PATH}'")
        return

    grayscale_image = cv2.cvtColor(original_image, cv2.COLOR_BGR2GRAY)

    sketch_mask = create_sketch_mask(
        grayscale_image,
        k_median_size=MEDIAN_KERNEL_SIZE,
        k_laplace_size=LAPLACIAN_KERNEL_SIZE,
        threshold_value=THRESHOLD_VALUE
    )

    color_painting = create_color_painting(
        original_image,
        d_size=D_SIZE,
        sigma_color=SIGMA_COLOR,
        sigma_space=SIGMA_SPACE,
        repetition_count=REPETITION_COUNT,
        downsample_factor=DOWNSAMPLE_FACTOR
    )

   
    final_cartoon_image = combine_with_sketch(color_painting, sketch_mask)

    
    print("Done! Displaying results.")
    cv2.imshow("1. Original Image", original_image)
    cv2.imshow("2. Sketch Mask (White Edges)", sketch_mask)
    cv2.imshow("3. Color Painting", color_painting)
    cv2.imshow("4. Final Cartoon Output", final_cartoon_image)

    cv2.imwrite("final_cartoon_image.jpg", final_cartoon_image)
    cv2.imwrite("sketch_mask.png", sketch_mask)
    print("\nFinal image saved as 'final_cartoon_image.jpg'.")
    print("Mask saved as 'sketch_mask.png'.")
    print("Press any key to close all windows.")
    
    cv2.waitKey(0)
    cv2.destroyAllWindows()

if __name__ == "__main__":
    main()

Loading images...
Applying Median Filter...
Applying Laplacian Filter...
Applying Threshold...
Creating Color Painting...
Combining painting and sketch...
Done! Displaying results.

Final image saved as 'final_cartoon_image.jpg'.
Mask saved as 'sketch_mask.png'.
Press any key to close all windows.
